# Lab Activity 4: Solving Nonlinear Equations with SciPy
**Course:** CSE473: Computational Intelligence — Mechatronics Engineering and Automation Program
**Prepares you for:** Lab Assignment 04 — Solving Nonlinear Equations with SciPy

## 🎯 Learning objectives
By the end of this lab you will be able to:
- Implement a nonlinear system $g_1 = g_2 = g_3 = 0$ as element-wise numpy functions
- Package the residuals into a vector and **scalarize**: $F = \tfrac{1}{2}\sum_i g_i^2$
- Explain why minimizing $F$ finds roots — $F \geq 0$ and $F = 0$ ⟺ $x$ is a solution
- Drive `scipy.optimize.minimize` to a (near-)zero residual, choosing methods and options
- Report the solution AND the optimizer settings, and probe the sensitivity to the starting point

⏱ **Estimated time: ~45 minutes**

## How this lab works
- The lab is split into **Parts**; each Part teaches one topic.
- Each Part starts with a short explanation plus a runnable **✏️ Worked example** — run it, tweak it, break it.
- Then you solve **🎯 Problems**. Read the task, write your code in the starter cell, and try it before opening any hints.
- Stuck? Open the **💡 Hint** blocks below each problem — Hint 1 is a nudge, Hint 2 names the approach. They get more specific as you go.
- Verify yourself with the **🧪 Self-check** cells — they run deterministic checks and fail with guiding messages until your solution is right.
- Truly stuck? The **✅ Reveal solution** block at the bottom of each hint section shows full working code.

In [ ]:
import numpy as np
from scipy.optimize import minimize
import scipy

print("NumPy version:", np.__version__)
print("SciPy version:", scipy.__version__)
print("Setup OK — NumPy and SciPy are ready.")

## Part 1: The nonlinear system as vector functions (≈10 min)

Lab Assignment 04 asks for a root of three nonlinear equations in three unknowns $x = (x_1, x_2, x_3)$:

$$g_1 = 3x_1 - \cos(x_2 x_3) - 0.5 = 0$$
$$g_2 = x_1^2 - 81(x_2 + 0.1)^2 + \sin(x_3) + 1.06 = 0$$
$$g_3 = e^{-x_1 x_2} + 20x_3 + \tfrac{10\pi - 3}{3} = 0$$

- Implement each $g_i$ as an **element-wise** function (`np.cos`, `np.sin`, `np.exp`) — the same code must accept scalars and arrays.
- A handy self-check: at the origin, $g_1 = -1.5$, $g_2 = 0.25$, $g_3 = 1 + \tfrac{10\pi - 3}{3}$ — compute these by hand once so you can spot formula typos.
- $\cos(x_2 x_3)$ is the cosine of the **product**; $e^{-x_1 x_2}$ is the exponential of the **negative product**.

In [ ]:
# --- Worked example: one equation, evaluated several ways ---
def g1_example(x1, x2, x3):
    return 3 * x1 - np.cos(x2 * x3) - 0.5

print("g1(0.5, 0, 0) =", g1_example(0.5, 0.0, 0.0), "  <- 3*0.5 - cos(0) - 0.5 = 0")
print("g1(0, 0, 0)   =", g1_example(0.0, 0.0, 0.0), " <- -cos(0) - 0.5 = -1.5")

# numpy functions are element-wise, so whole arrays work too:
print("g1 on a vector of x1:", g1_example(np.array([0.0, 0.5, 1.0]), 0.0, 0.0))

## 🎯 Problem 4.1 — Implement g1, g2, g3

**Given:** the three equations above.

**Required:** write three element-wise functions:

- `g1(x1, x2, x3)` — $3x_1 - \cos(x_2 x_3) - 0.5$
- `g2(x1, x2, x3)` — $x_1^2 - 81(x_2 + 0.1)^2 + \sin(x_3) + 1.06$
- `g3(x1, x2, x3)` — $e^{-x_1 x_2} + 20x_3 + \tfrac{10\pi - 3}{3}$ (use `np.pi`)

**Expected output:** the values at the origin quoted in the concept text; $g_1$ linear in $x_1$ with slope 3; $g_2$ symmetric about $x_2 = -0.1$; $g_3$ shifting by $20\,\Delta x_3$ when only $x_3$ moves.

In [ ]:
def g1(x1, x2, x3):
    """g1 = 3*x1 - cos(x2*x3) - 0.5 (element-wise)."""
    # TODO: Your code here
    pass

def g2(x1, x2, x3):
    """g2 = x1^2 - 81*(x2 + 0.1)^2 + sin(x3) + 1.06 (element-wise)."""
    # TODO: Your code here
    pass

def g3(x1, x2, x3):
    """g3 = exp(-x1*x2) + 20*x3 + (10*pi - 3)/3 (element-wise)."""
    # TODO: Your code here
    pass

# Demo call
demo_g1 = g1(0.0, 0.0, 0.0)
if demo_g1 is not None:
    print("g1(0,0,0) =", demo_g1)
    print("g2(0,0,0) =", g2(0.0, 0.0, 0.0))
    print("g3(0,0,0) =", g3(0.0, 0.0, 0.0))
else:
    print("Implement g1, g2, g3 to power this demo.")

<details>
<summary>💡 Hint 1 — translate term by term</summary>

Three tiny numpy expressions, one per equation. np.cos of the PRODUCT x2*x3, np.sin(x3), np.exp of the negative product -x1*x2, and remember np.pi for the last constant.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
def g1(x1, x2, x3):
    return 3 * x1 - np.cos(x2 * x3) - 0.5
def g2(x1, x2, x3):
    return x1 ** 2 - 81 * (x2 + 0.1) ** 2 + np.sin(x3) + 1.06
def g3(x1, x2, x3):
    return np.exp(-x1 * x2) + 20 * x3 + (10 * np.pi - 3) / 3
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def g1(x1, x2, x3):
    """g1 = 3*x1 - cos(x2*x3) - 0.5 (element-wise)."""
    return 3 * x1 - np.cos(x2 * x3) - 0.5

def g2(x1, x2, x3):
    """g2 = x1^2 - 81*(x2 + 0.1)^2 + sin(x3) + 1.06 (element-wise)."""
    return x1 ** 2 - 81 * (x2 + 0.1) ** 2 + np.sin(x3) + 1.06

def g3(x1, x2, x3):
    """g3 = exp(-x1*x2) + 20*x3 + (10*pi - 3)/3 (element-wise)."""
    return np.exp(-x1 * x2) + 20 * x3 + (10 * np.pi - 3) / 3
```
</details>

In [ ]:
# 🧪 Self-check for Problem 4.1
a1 = g1(0.0, 0.0, 0.0)
assert a1 is not None, "❌ g1 returned None — replace the 'pass' stubs. See Hint 2 in Part 1."
assert abs(float(a1) + 1.5) < 1e-12, "❌ g1(0,0,0) = 3*0 - cos(0) - 0.5 = -1.5. Check the constant and the cosine term."
assert abs(float(g2(0.0, 0.0, 0.0)) - 0.25) < 1e-12, "❌ g2(0,0,0) = -81*(0.1)^2 + 1.06 = 0.25. Check the -81, the +0.1 and the +1.06."
assert abs(float(g3(0.0, 0.0, 0.0)) - (1.0 + (10 * np.pi - 3) / 3)) < 1e-12, "❌ g3(0,0,0) = exp(0) + 0 + (10*pi - 3)/3. Did you use np.pi?"
assert abs(float(g1(1.0, 0.2, 0.3)) - float(g1(0.0, 0.2, 0.3)) - 3.0) < 1e-12, "❌ g1 must be linear in x1 with slope 3."
assert abs(float(g2(0.0, 0.9, 0.0)) - float(g2(0.0, -1.1, 0.0))) < 1e-12, "❌ g2 is symmetric about x2 = -0.1 (the squared term) — check (x2 + 0.1)."
assert abs(float(g3(0.0, 0.0, 0.1)) - float(g3(0.0, 0.0, 0.0)) - 2.0) < 1e-12, "❌ g3 must shift by 20*dx3 when only x3 changes — check the 20*x3 term."
rng4 = np.random.default_rng(0)
A4, B4, C4 = rng4.normal(size=(6, 4)), rng4.normal(size=(6, 4)), rng4.normal(size=(6, 4))
assert getattr(g1(A4, B4, C4), "shape", None) == (6, 4), "❌ g1 must work element-wise on equal-shape arrays — use np.cos/np.exp, not math.cos/math.exp."
print("✅ Problem 4.1 passed — the system is implemented.")

## 🎯 Problem 4.2 — Residual vector and objective

**Given:** your $g_1, g_2, g_3$.

**Required:** write:

- `equations_vector(x)` — takes ONE array `x = [x1, x2, x3]`, returns `np.array([g1, g2, g3])` evaluated at `x`,
- `F_objective(x)` — returns $F = \tfrac{1}{2}\,(g_1^2 + g_2^2 + g_3^2)$ as a **plain float**.

These two bridge "the system" and "the optimizer": the assignment grades your answer with exactly `equations_vector(solution)`.

In [ ]:
def equations_vector(x):
    """Return [g1, g2, g3] evaluated at x = [x1, x2, x3]."""
    # TODO: Your code here
    pass

def F_objective(x):
    """F = 1/2 * (g1^2 + g2^2 + g3^2) at x = [x1, x2, x3]."""
    # TODO: Your code here
    pass

# Demo call
demo_v = equations_vector(np.array([0.0, 0.0, 0.0]))
if demo_v is not None:
    print("equations_vector(0,0,0) =", demo_v)
    print("F at the origin =", F_objective(np.array([0.0, 0.0, 0.0])))
else:
    print("Implement equations_vector to power this demo.")

<details>
<summary>💡 Hint 1 — unpack x first</summary>

Both functions receive ONE array x. Unpack it (x1, x2, x3 = x), then either build the array of the three g values, or square and halve that same vector.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
def equations_vector(x):
    x1, x2, x3 = x
    return np.array([g1(x1, x2, x3), g2(x1, x2, x3), g3(x1, x2, x3)])

def F_objective(x):
    g = equations_vector(x)
    return float(0.5 * (g @ g))
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def equations_vector(x):
    """Return [g1, g2, g3] evaluated at x = [x1, x2, x3]."""
    x1, x2, x3 = x
    return np.array([g1(x1, x2, x3), g2(x1, x2, x3), g3(x1, x2, x3)])

def F_objective(x):
    """F = 1/2 * (g1^2 + g2^2 + g3^2) at x = [x1, x2, x3]."""
    g = equations_vector(x)
    return float(0.5 * (g @ g))
```
</details>

In [ ]:
# 🧪 Self-check for Problem 4.2
v0 = equations_vector(np.zeros(3))
assert v0 is not None, "❌ equations_vector returned None — replace the 'pass'. See Hint 2 in Part 1."
v0 = np.asarray(v0, dtype=float)
assert v0.shape == (3,), f"❌ equations_vector must return a 3-element array, got shape {v0.shape}."
assert np.allclose(v0, [-1.5, 0.25, 1.0 + (10 * np.pi - 3) / 3]), "❌ equations_vector(0,0,0) should match the g_i values you verified in Problem 4.1."
f0 = F_objective(np.zeros(3))
assert f0 is not None, "❌ F_objective returned None — replace the 'pass'."
assert isinstance(f0, float), "❌ F_objective should return a plain float — wrap with float(...)."
assert abs(f0 - 0.5 * float(v0 @ v0)) < 1e-12, "❌ F must equal 1/2 * ||equations_vector(x)||^2."
rng42 = np.random.default_rng(1)
for _ in range(5):
    xt = rng42.normal(size=3)
    vt = np.asarray(equations_vector(xt), dtype=float)
    ft = F_objective(xt)
    assert ft >= 0.0, "❌ F is a sum of squares — it can never be negative."
    assert abs(ft - 0.5 * float(vt @ vt)) < 1e-12, "❌ F(x) must always equal 1/2 * (g1^2 + g2^2 + g3^2) — recompute it from equations_vector."
print("✅ Problem 4.2 passed — objective and residual vector agree.")

## Part 2: Scalarization — turning roots into minima (≈8 min)

No root-finder is needed if you reformulate. Define

$$F(x) = \tfrac{1}{2}\,\|g(x)\|^2 = \tfrac{1}{2}\left(g_1^2 + g_2^2 + g_3^2\right) \geq 0$$

Two facts make this work:

1. $F$ is a sum of squares, so $F(x) \geq 0$ **everywhere**, and
2. $F(x) = 0$ **exactly where all three** $g_i = 0$ — i.e. at the roots.

So *any* minimizer of $F$ that reaches $F \approx 0$ is a root of the system. A general smooth minimizer (`scipy.optimize.minimize`) does the job — that is the whole trick of the assignment.

In [ ]:
# --- Worked example: F along the line (t, 0, 0) ---
def _g1(x1, x2, x3): return 3 * x1 - np.cos(x2 * x3) - 0.5
def _g2(x1, x2, x3): return x1 ** 2 - 81 * (x2 + 0.1) ** 2 + np.sin(x3) + 1.06
def _g3(x1, x2, x3): return np.exp(-x1 * x2) + 20 * x3 + (10 * np.pi - 3) / 3
def F_example(x):
    x1, x2, x3 = x
    return 0.5 * (_g1(x1, x2, x3) ** 2 + _g2(x1, x2, x3) ** 2 + _g3(x1, x2, x3) ** 2)

probe = [F_example(np.array([a, b, c])) for a in (0.0, 1.0) for b in (-1.0, 0.5) for c in (-0.5, 0.2)]
print("F is never negative:", all(f >= 0.0 for f in probe))
print("F along (t, 0, 0):")
for t in np.linspace(0.0, 1.0, 6):
    print(f"  t={t:.1f}  F={F_example(np.array([t, 0.0, 0.0])):12.6f}")
print("The valley floor touches ~0 where the line crosses a root — that is what minimize will find.")

## 🎯 Problem 4.3 — A residual report

**Given:** a point `x` and your `F_objective` / `equations_vector`.

**Required:** write `residual_report(x)` returning a dict with keys:

- `'F'` — the objective value at `x` (float),
- `'residuals'` — the 3-element residual vector (`equations_vector(x)`),
- `'residual_norm'` — $\|g(x)\| = \sqrt{g_1^2 + g_2^2 + g_3^2}$ as a float.

**Expected output:** the three entries always satisfy $F = \tfrac{1}{2}\,\text{residual\_norm}^2$ — the self-check verifies exactly that identity.

In [ ]:
def residual_report(x):
    """Package the objective, residuals and residual norm at x.

    Args:
        x: array-like of shape (3,)

    Returns:
        dict with keys 'F', 'residuals', 'residual_norm'
    """
    # TODO: Your code here
    pass

# Demo call
demo_rep4 = residual_report(np.array([0.0, 0.0, 0.0]))
if demo_rep4 is not None:
    print("report at the origin:", demo_rep4)
else:
    print("Implement residual_report to power this demo.")

<details>
<summary>💡 Hint 1 — reuse, don't recompute</summary>

Call equations_vector ONCE and derive everything from that vector: F is half its squared length, residual_norm its length (np.linalg.norm).
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
def residual_report(x):
    g = equations_vector(x)
    norm = float(np.linalg.norm(g))
    return {"F": F_objective(x), "residuals": g, "residual_norm": norm}
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def residual_report(x):
    """Package the objective, residuals and residual norm at x."""
    g = equations_vector(x)
    norm = float(np.linalg.norm(g))
    return {"F": F_objective(x), "residuals": g, "residual_norm": norm}
```
</details>

In [ ]:
# 🧪 Self-check for Problem 4.3
rep0 = residual_report(np.array([0.0, 0.0, 0.0]))
assert rep0 is not None, "❌ residual_report returned None — replace the 'pass'. See Hint 2 in Part 2."
for k in ("F", "residuals", "residual_norm"):
    assert k in rep0, f"❌ Report dict is missing '{k}'."
assert isinstance(rep0["F"], float) and isinstance(rep0["residual_norm"], float), "❌ 'F' and 'residual_norm' must be plain floats."
res_arr = np.asarray(rep0["residuals"], dtype=float)
assert res_arr.shape == (3,), "❌ 'residuals' must be the 3-element vector from equations_vector."
assert abs(rep0["F"] - 0.5 * rep0["residual_norm"] ** 2) < 1e-12, "❌ The identity F = 1/2 * residual_norm^2 must hold — compute both from the same g values."
assert rep0["residual_norm"] >= 0.0, "❌ A norm is never negative — check np.linalg.norm(residuals)."
assert np.allclose(res_arr, equations_vector(np.zeros(3))), "❌ 'residuals' must equal equations_vector(x) at the same x."
xr = np.random.default_rng(2).normal(size=3)
repr_ = residual_report(xr)
assert np.isclose(repr_["residual_norm"], float(np.linalg.norm(equations_vector(xr)))), "❌ residual_norm must be the 2-norm of equations_vector(x) at ANY x."
print("✅ Problem 4.3 passed — you can grade any candidate solution.")

## Part 3: scipy.optimize.minimize (≈12 min)

`scipy.optimize.minimize(fun, x0, method=..., options=...)` is the general workhorse:

- `fun` — the scalar function to minimize (here: `F_objective`),
- `x0` — the starting point: iterative methods need a guess, and different guesses can end at different minima,
- `method` — `"BFGS"` (default; quasi-Newton), `"CG"` (conjugate gradient), `"Nelder-Mead"` (derivative-free simplex),
- `options` — per-method controls such as `{"gtol": ..., "maxiter": ...}`.

The result object carries `result.x` (the minimizer), `result.fun` (final $F$), `result.success` and `result.message`. **Caveat:** with very tight tolerances BFGS can stop with `success=False` ("precision loss") while `result.fun` is already $\sim 10^{-14}$ — the answer can be right even when the flag says otherwise. Judge a solve by its **residual**, not only the flag.

In [ ]:
# --- Worked example: minimize a simple quadratic two ways ---
def quad(x):
    return (x[0] - 2.0) ** 2 + (x[1] + 1.0) ** 2     # minimum at (2, -1)

r_bfgs = minimize(quad, np.array([0.0, 0.0]), method="BFGS")
print("BFGS       : x =", np.round(r_bfgs.x, 6), "| success =", r_bfgs.success, "| F =", r_bfgs.fun)

r_nm = minimize(quad, np.array([5.0, 5.0]), method="Nelder-Mead",
                options={"xatol": 1e-10, "fatol": 1e-12})
print("Nelder-Mead: x =", np.round(r_nm.x, 6), "| success =", r_nm.success, "| F =", r_nm.fun)
print("useful result fields:", [f for f in ("x", "fun", "success", "message", "nit") if hasattr(r_bfgs, f)])

## 🎯 Problem 4.4 — A reusable minimization wrapper

**Given:** a scalar function `fun`, a starting point `x0`, a `method` and optional `options`.

**Required:** write `run_minimization(fun, x0, method="BFGS", options=None)` that calls `scipy.optimize.minimize(fun, x0, method=method, options=options)` and **returns the result object** (callers read `.x`, `.fun`, `.success`).

**Expected output:** on `quad(x) = (x[0]-2)^2 + (x[1]+1)^2` from any reasonable start, BFGS converges with `success=True` and `fun` near zero.

In [ ]:
def run_minimization(fun, x0, method="BFGS", options=None):
    """Minimize `fun` starting from `x0` and return the scipy result.

    Args:
        fun: scalar function of a 1-D array
        x0: starting point (array-like)
        method: optimizer method, e.g. "BFGS", "CG", "Nelder-Mead"
        options: optional dict of method options (e.g. {"gtol": 1e-10})

    Returns:
        scipy.optimize.OptimizeResult
    """
    # TODO: Your code here
    pass

# Demo call
demo_r = run_minimization(lambda v: (v[0] - 1.0) ** 2, np.array([0.0]))
if demo_r is not None:
    print("minimizer:", demo_r.x, "| success:", demo_r.success)
else:
    print("Implement run_minimization to power this demo.")

<details>
<summary>💡 Hint 1 — pass everything through</summary>

The wrapper adds nothing but defaults: hand fun, x0, method and options straight to scipy.optimize.minimize and return its result. Convert x0 with np.asarray(..., dtype=float) to be safe.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
return minimize(fun, np.asarray(x0, dtype=float), method=method, options=options)
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def run_minimization(fun, x0, method="BFGS", options=None):
    """Minimize `fun` starting from `x0`; return the scipy result."""
    return minimize(fun, np.asarray(x0, dtype=float), method=method, options=options)
```
</details>

In [ ]:
# 🧪 Self-check for Problem 4.4
def quad44(x):
    return (x[0] - 2.0) ** 2 + (x[1] + 1.0) ** 2

r44 = run_minimization(quad44, np.array([0.0, 0.0]), method="BFGS")
assert r44 is not None, "❌ run_minimization returned None — replace the 'pass'. See Hint 2 in Part 3."
assert hasattr(r44, "x") and hasattr(r44, "fun") and hasattr(r44, "success"), "❌ Return the scipy result object itself (it carries .x, .fun, .success)."
assert np.asarray(r44.x).shape == (2,), "❌ The minimizer should be a 2-element vector for this 2-D test function."
assert bool(r44.success), "❌ BFGS on a convex quadratic should converge with success=True — did you pass x0 and method correctly?"
assert float(r44.fun) < 1e-8, "❌ The final objective should be nearly zero at the quadratic's minimum."
assert np.allclose(r44.x, [2.0, -1.0], atol=1e-3), "❌ The minimizer of (x0-2)^2 + (x1+1)^2 is (2, -1) — check the argument order of minimize."
r44b = run_minimization(quad44, [5.0, 5.0], method="BFGS")
assert r44b is not None and float(r44b.fun) < 1e-8, "❌ A list x0 must work too (minimize accepts array-like starts)."
print("✅ Problem 4.4 passed — minimize is under your control.")

## 🎯 Problem 4.5 — How sensitive is the start?

**Given:** a list of starting points `x0_list`.

**Required:** write `sweep_starting_points(x0_list, method="BFGS", options=None)` that minimizes **your** `F_objective` from every start and returns a list of dicts — one per start — with keys `'x0'`, `'success'`, `'F_min'`, `'residual_norm'`, where `residual_norm` is $\|g(x^*)\|$ from your `equations_vector` at the found minimizer.

**Expected output:** for this system BFGS drives the residual below $10^{-5}$ from all reasonable starts — but runs may disagree on `success` and take different paths. That is the lesson: **always report the residual**.

In [ ]:
def sweep_starting_points(x0_list, method="BFGS", options=None):
    """Minimize F_objective from each starting point; report the outcome.

    Args:
        x0_list: iterable of starting points (each array-like of length 3)
        method: optimizer method passed to scipy.optimize.minimize
        options: optional dict of method options

    Returns:
        list of dicts with keys 'x0', 'success', 'F_min', 'residual_norm'
    """
    # TODO: Your code here
    pass

# Demo call
demo_sw = sweep_starting_points([(0.0, 0.0, 0.0), (1.0, -1.0, 0.5)])
if demo_sw is not None:
    for row in demo_sw:
        print(row)
else:
    print("Implement sweep_starting_points to power this demo.")

<details>
<summary>💡 Hint 1 — loop over starts</summary>

Each iteration: minimize F_objective from that x0, then grade the minimizer with equations_vector — F_min from result.fun, residual_norm from the norm of g(result.x).
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
runs = []
for x0 in x0_list:
    result = minimize(F_objective, np.asarray(x0, dtype=float), method=method, options=options)
    runs.append({"x0": x0, "success": bool(result.success),
                 "F_min": float(result.fun),
                 "residual_norm": float(np.linalg.norm(equations_vector(result.x)))})
return runs
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def sweep_starting_points(x0_list, method="BFGS", options=None):
    """Minimize F_objective from each starting point; report the outcome."""
    runs = []
    for x0 in x0_list:
        result = minimize(F_objective, np.asarray(x0, dtype=float),
                          method=method, options=options)
        runs.append({"x0": x0,
                     "success": bool(result.success),
                     "F_min": float(result.fun),
                     "residual_norm": float(np.linalg.norm(equations_vector(result.x)))})
    return runs
```
</details>

In [ ]:
# 🧪 Self-check for Problem 4.5
starts = [(0.0, 0.0, 0.0), (1.0, -1.0, 0.5), (-1.0, 1.0, -0.5)]
sw = sweep_starting_points(starts, method="BFGS")
assert sw is not None, "❌ sweep_starting_points returned None — replace the 'pass'. See Hint 2 in Part 3."
assert len(sw) == len(starts), f"❌ Expected one report per start ({len(starts)}), got {len(sw)}."
for row in sw:
    assert set(row.keys()) >= {"x0", "success", "F_min", "residual_norm"}, f"❌ Each report needs keys x0, success, F_min, residual_norm — got {sorted(row.keys())}."
    assert np.isfinite(row["F_min"]) and row["F_min"] >= 0.0, "❌ F_min must be a finite, non-negative number."
    assert np.isfinite(row["residual_norm"]) and row["residual_norm"] >= 0.0, "❌ residual_norm must be a finite, non-negative number."
best5 = min(row["residual_norm"] for row in sw)
assert best5 < 1e-5, "❌ At least one BFGS run should reach ||g(x*)|| < 1e-5 from these starts — check F_objective and equations_vector."
sw2 = sweep_starting_points(starts, method="BFGS")
assert [r["F_min"] for r in sw2] == [r["F_min"] for r in sw], "❌ Same starts must reproduce identical results — keep everything deterministic."
print("✅ Problem 4.5 passed — starting points swept, residuals reported.")

## Part 4: Solve the assignment system & mini-challenge (≈12 min)

You now own every piece: equations → `equations_vector`, scalarization → `F_objective`, solver → `run_minimization`, grading → `residual_report`. The assignment's pipeline is: pick settings, minimize from $x_0 = (0, 0, 0)$, then report the solution, $F_{\min}$, the residual **and the settings you used**.

**Mini-challenge:** try several methods/starting points and report the best run — exactly what a careful engineer (and the assignment rubric) wants to see.

In [ ]:
# --- Worked example: the full solve, with the success-flag lesson ---
r_default = minimize(F_example, np.array([0.0, 0.0, 0.0]), method="BFGS")
res_def = np.array([_g1(*r_default.x), _g2(*r_default.x), _g3(*r_default.x)])
print("default options: success =", r_default.success,
      "| F = %.3e" % r_default.fun, "| ||g|| = %.3e" % np.linalg.norm(res_def))

r_tight = minimize(F_example, np.array([0.0, 0.0, 0.0]), method="BFGS",
                   options={"gtol": 1e-10, "maxiter": 500})
res_tight = np.array([_g1(*r_tight.x), _g2(*r_tight.x), _g3(*r_tight.x)])
print("tight gtol    : success =", r_tight.success,
      "| F = %.3e" % r_tight.fun, "| ||g|| = %.3e" % np.linalg.norm(res_tight))
print("Lesson: the tight run reports success=False (precision loss) yet F ~ 1e-14 — judge by the residual.")

## 🎯 Problem 4.6 — Solve the system

**Given:** everything from Parts 1–3.

**Required:** write `solve_system(x0=(0.0, 0.0, 0.0), method="BFGS", options=None)` returning a dict with keys:

- `'solution'` — the minimizer (`result.x`),
- `'F_min'` — final objective (float),
- `'success'` — the optimizer's flag (bool),
- `'residuals'` — `equations_vector(solution)`,
- `'residual_norm'` — $\|g(\text{solution})\|$ (float).

**Expected output:** a correct solve drives $\|g\|$ below $10^{-5}$ (typically $\sim 10^{-7}$ with default BFGS). **Do not require `success=True`** — with tight tolerances the flag may be False while the residual is excellent.

In [ ]:
def solve_system(x0=(0.0, 0.0, 0.0), method="BFGS", options=None):
    """Solve g1 = g2 = g3 = 0 by minimizing F = 1/2 * ||g||^2.

    Args:
        x0: starting point (default (0, 0, 0), the assignment's guess)
        method: optimizer method
        options: optional dict of method options

    Returns:
        dict with keys 'solution', 'F_min', 'success', 'residuals', 'residual_norm'
    """
    # TODO: Your code here
    pass

# Demo call
demo_sol = solve_system()
if demo_sol is not None:
    print("solution:", demo_sol["solution"])
    print("F_min = %.3e | residual_norm = %.3e | success = %s" %
          (demo_sol["F_min"], demo_sol["residual_norm"], demo_sol["success"]))
else:
    print("Implement solve_system to power this demo.")

<details>
<summary>💡 Hint 1 — chain your own tools</summary>

You already own a wrapper (run_minimization) and a grader (residual_report): minimize, then build the dict from result.x / result.fun / result.success plus residual_report(result.x).
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
result = minimize(F_objective, np.asarray(x0, dtype=float), method=method, options=options)
residuals = equations_vector(result.x)
return {"solution": result.x, "F_min": float(result.fun),
        "success": bool(result.success), "residuals": residuals,
        "residual_norm": float(np.linalg.norm(residuals))}
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def solve_system(x0=(0.0, 0.0, 0.0), method="BFGS", options=None):
    """Solve g1 = g2 = g3 = 0 by minimizing F = 1/2 * ||g||^2."""
    result = minimize(F_objective, np.asarray(x0, dtype=float),
                      method=method, options=options)
    residuals = equations_vector(result.x)
    return {"solution": result.x,
            "F_min": float(result.fun),
            "success": bool(result.success),
            "residuals": residuals,
            "residual_norm": float(np.linalg.norm(residuals))}
```
</details>

In [ ]:
# 🧪 Self-check for Problem 4.6
sol = solve_system()
assert sol is not None, "❌ solve_system returned None — replace the 'pass'. See Hint 2 in Part 4."
for k in ("solution", "F_min", "success", "residuals", "residual_norm"):
    assert k in sol, f"❌ Dict is missing '{k}'."
xs = np.asarray(sol["solution"], dtype=float)
assert xs.shape == (3,), f"❌ solution must be a 3-element vector, got shape {xs.shape}."
assert np.isfinite(xs).all(), "❌ solution must be finite — did minimize actually converge? Check F_objective."
assert sol["residual_norm"] < 1e-5, "❌ ||g(x*)|| should be below 1e-5 — the solve did not reach a root. Check that F_objective is 1/2*(g1^2 + g2^2 + g3^2)."
assert abs(sol["F_min"] - 0.5 * sol["residual_norm"] ** 2) < 1e-12, "❌ F_min must equal 1/2 * residual_norm^2 — both must come from the same final point."
assert np.allclose(np.asarray(sol["residuals"], dtype=float), equations_vector(xs), atol=1e-12), "❌ 'residuals' must be equations_vector(solution)."
sol_b = solve_system(method="CG")
assert sol_b is not None and sol_b["residual_norm"] < 1e-5, "❌ CG from (0,0,0) should also reach ||g|| < 1e-5 — check your objective again."
print("✅ Problem 4.6 passed — the nonlinear system is solved.")

## 🎯 Problem 4.7 — Mini-challenge: a method study

**Given:** your full toolkit.

**Required:** write `method_study(x0_list=((0, 0, 0), (1, -1, 0.5), (-1, 1, -0.5)), methods=("BFGS", "Nelder-Mead"))` that runs **every (start, method) pair** and returns a dict:

- `'runs'` — a list of dicts, one per run, with keys `'x0'`, `'method'`, `'F_min'`, `'residual_norm'`, `'success'`,
- `'best'` — the run dict with the **smallest residual_norm**.

**Expected output:** `best['residual_norm']` below $10^{-5}$ — BFGS nails this system from anywhere reasonable, while Nelder-Mead (derivative-free) usually stops at a looser residual. Reporting settings *and* residuals per run is exactly what Lab Assignment 04's rubric asks for.

In [ ]:
def method_study(x0_list=((0.0, 0.0, 0.0), (1.0, -1.0, 0.5), (-1.0, 1.0, -0.5)),
                 methods=("BFGS", "Nelder-Mead")):
    """Run every (start, method) pair; collect reports and the best run.

    Args:
        x0_list: iterable of starting points
        methods: iterable of method names for scipy.optimize.minimize

    Returns:
        dict with keys 'runs' (list of dicts) and 'best' (one run dict)
    """
    # TODO: Your code here
    pass

# Demo call
demo_study = method_study()
if demo_study is not None:
    print("runs:", len(demo_study["runs"]))
    print("best residual:", demo_study["best"]["residual_norm"])
else:
    print("Implement method_study to power this demo.")

<details>
<summary>💡 Hint 1 — two nested loops</summary>

Outer loop over methods, inner loop over starts; collect one run dict per pair. best is the run with the smallest residual_norm (min(..., key=...)).
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
runs = []
for method in methods:
    for x0 in x0_list:
        result = minimize(F_objective, np.asarray(x0, dtype=float), method=method)
        runs.append({"x0": x0, "method": method, "F_min": float(result.fun),
                     "residual_norm": float(np.linalg.norm(equations_vector(result.x))),
                     "success": bool(result.success)})
best = min(runs, key=lambda r: r["residual_norm"])
return {"runs": runs, "best": best}
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def method_study(x0_list=((0.0, 0.0, 0.0), (1.0, -1.0, 0.5), (-1.0, 1.0, -0.5)),
                 methods=("BFGS", "Nelder-Mead")):
    """Run every (start, method) pair; collect reports and the best run."""
    runs = []
    for method in methods:
        for x0 in x0_list:
            result = minimize(F_objective, np.asarray(x0, dtype=float), method=method)
            runs.append({"x0": x0,
                         "method": method,
                         "F_min": float(result.fun),
                         "residual_norm": float(np.linalg.norm(equations_vector(result.x))),
                         "success": bool(result.success)})
    best = min(runs, key=lambda r: r["residual_norm"])
    return {"runs": runs, "best": best}
```
</details>

In [ ]:
# 🧪 Self-check for Problem 4.7
study = method_study()
assert study is not None, "❌ method_study returned None — replace the 'pass'. See Hint 2 in Part 4."
assert set(study.keys()) >= {"runs", "best"}, f"❌ Dict needs keys 'runs' and 'best' — got {sorted(study.keys())}."
assert len(study["runs"]) == 6, f"❌ Expected 3 starts x 2 methods = 6 runs, got {len(study['runs'])}."
methods_seen = {r["method"] for r in study["runs"]}
assert methods_seen == {"BFGS", "Nelder-Mead"}, f"❌ Both methods must appear in the runs — saw {methods_seen}."
for r in study["runs"]:
    assert set(r.keys()) >= {"x0", "method", "F_min", "residual_norm", "success"}, f"❌ Each run needs x0, method, F_min, residual_norm, success — got {sorted(r.keys())}."
    assert np.isfinite(r["F_min"]) and r["F_min"] >= 0.0, "❌ F_min must be finite and non-negative."
best7 = study["best"]
assert best7 in study["runs"], "❌ 'best' must be one of the run dicts."
assert best7["residual_norm"] == min(r["residual_norm"] for r in study["runs"]), "❌ 'best' must be the run with the smallest residual_norm."
assert best7["residual_norm"] < 1e-5, "❌ The best run should reach ||g|| < 1e-5 — check F_objective and the BFGS runs."
print("✅ Problem 4.7 passed — settings swept, best run reported. Lab Assignment 04 awaits!")

## 🎉 You've completed Lab Activity 4

You have mastered:
- Implementing a nonlinear system element-wise ($g_1, g_2, g_3$) and packaging it as a residual vector
- Scalarization: $F = \tfrac{1}{2}\|g\|^2$ turns "solve $g = 0$" into "minimize $F$", with $F \geq 0$ and $F = 0$ ⟺ a root
- Driving `scipy.optimize.minimize` with method and `options` choices — and reading `result.x` / `.fun` / `.success` / `.message`
- Judging a solve by its **residual**, not just the success flag
- Sweeping starting points and methods, and reporting the best run with its settings

**You are now ready for Lab Assignment 04 on the course portal — the assignment asks for the same techniques without hints.**

💡 **Tip:** restart the kernel and run every cell top-to-bottom once more — each 🧪 self-check should print ✅, and the residual-report habit is what the assignment rubric actually grades.